In [1]:
!pip install llmlingua -q

In [2]:
!pip install deepeval -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 864.6/864.6 kB 15.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.3/228.3 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 2.3 MB/s eta 0:00:00


In [3]:
from llmlingua import PromptCompressor
import pandas as pd

In [4]:
# from huggingface_hub import InferenceClient
# from dotenv import load_dotenv
# import os

# load_dotenv('.env')

# hf_token = os.getenv('HF_token')
# client = InferenceClient(token=)

Сжатие в 5 раз

In [5]:
llm_lingua = PromptCompressor(
    model_name = 'microsoft/llmlingua-2-xlm-roberta-large-meetingbank',
    use_llmlingua2 = True
)

config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [6]:
prompt = 'Original Text: Item 15, report from City Manager Recommendation to adopt three resolutions. First, to join the Victory Pace program. Second, to join the California first program. And number three, consenting to to inclusion of certain properties within the jurisdiction in the California Hero program. It was emotion, motion, a second and public comment. CNN. Please cast your vote. Oh. Was your public comment? Yeah. Please come forward. I thank you, Mr. Mayor. Thank you. Members of the council. My name is Alex Mitchell. I represent the hero program. Just wanted to let you know that the hero program. Has been in California for the last three and a half years.'

In [7]:
compressed_prompt = llm_lingua.compress_prompt(prompt)

In [8]:
compressed_prompt['compressed_prompt']

'Text Item 15 report City Manager Recommendation adopt three resolutions First join Victory Pace program Second join California first program three inclusion properties jurisdiction California Hero program emotion, motion second public comment CNN cast vote public comment? come forward thank you Mr. Mayor Members council Alex Mitchell represent hero program California three and a half years'

In [9]:
# openai_key = os.getenv("API_KEY_GPT4")

In [10]:
openai_key = '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'

In [11]:
from openai import OpenAI

cliennt = OpenAI(api_key = openai_key)

model = 'GPT-3.5-Turbo-0613'

In [20]:
from deepeval.models.base_model import DeepEvalBaseLLM
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

class LLMLinguaGPT(DeepEvalBaseLLM):
    def __init__(self, compressor):
        openai_key = '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'
        self.client = OpenAI(
          base_url="https://openrouter.ai/api/v1",
          api_key=openai_key,
        )
        self.compressor = compressor
        self._model_name = 'openai/gpt-3.5-turbo-instruct'

    def load_model(self):
        return self.compressor

    def generate(self, prompt: str, schema=None):
        compressed_prompt = self.compressor.compress_prompt(prompt)['compressed_prompt']
        response = self.client.chat.completions.create(
            model="openai/gpt-3.5-turbo-0613",
            messages=[{"role": "user", "content": compressed_prompt}],
            temperature=0
        )
        return response.choices[0].message.content

    async def a_generate(self, prompt:str):
        return self.generate(prompt=prompt)

    def get_model_name(self):
        return "LLMLingua2"

In [21]:
from deepeval.benchmarks import BigBenchHard
from deepeval.benchmarks.tasks import BigBenchHardTask

benchmark = BigBenchHard(
    tasks=[BigBenchHardTask.BOOLEAN_EXPRESSIONS, BigBenchHardTask.CAUSAL_JUDGEMENT],
    enable_cot = False
)

In [22]:
model = LLMLinguaGPT(compressor=llm_lingua)

benchmark.evaluate(model=model)
print(benchmark.overall_score)

Processing boolean_expressions: 100%|██████████| 250/250 [54:08<00:00, 12.99s/it]  


Big Bench Hard Task Accuracy (task=boolean_expressions): 0.608


Processing causal_judgement: 100%|██████████| 187/187 [10:32<00:00,  3.38s/it]

Big Bench Hard Task Accuracy (task=causal_judgement): 0.5026737967914439
Overall Big Bench Hard Accuracy: 0.562929061784897
0.562929061784897


In [16]:
from deepeval.models.base_model import DeepEvalBaseLLM
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

class LLMLinguaGPT(DeepEvalBaseLLM):
    def __init__(self, compressor):
        openai_key = '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'
        self.client = OpenAI(
          base_url="https://openrouter.ai/api/v1",
          api_key=openai_key,
        )

    def load_model(self):
        return self.compressor

    def generate(self, prompt: str, schema=None):
        response = self.client.chat.completions.create(
            model="openai/gpt-3.5-turbo-0613",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return response.choices[0].message.content

    async def a_generate(self, prompt:str):
        return self.generate(prompt=prompt)

    def get_model_name(self):
        return "LLMLingua2"

In [14]:
model = LLMLinguaGPT(compressor=llm_lingua)

benchmark.evaluate(model=model)
print(benchmark.overall_score)

README.md: 0.00B [00:00, ?B/s]

boolean_expressions/test-00000-of-00001.(…):   0%|          | 0.00/4.52k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/250 [00:00<?, ? examples/s]

Processing boolean_expressions: 100%|██████████| 250/250 [56:22<00:00, 13.53s/it]  


Big Bench Hard Task Accuracy (task=boolean_expressions): 0.596


causal_judgement/test-00000-of-00001.par(…):   0%|          | 0.00/67.8k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/187 [00:00<?, ? examples/s]

Processing causal_judgement: 100%|██████████| 187/187 [12:21<00:00,  3.97s/it]

Big Bench Hard Task Accuracy (task=causal_judgement): 0.5026737967914439
Overall Big Bench Hard Accuracy: 0.5560640732265446
0.5560640732265446
